In [102]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
# NEW METHOD - search_experiments()
experiments = client.search_experiments()

for exp in experiments:
    print(f"Experiment: {exp.name} | ID: {exp.experiment_id}")

Experiment: nyc-taxi-experiment | ID: 1
Experiment: Default | ID: 0


In [103]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids=["1"], 
    filter_string="metrics.rmse < 6.8",
    run_view_type=ViewType.ACTIVE_ONLY, 
    max_results=5, 
    order_by=["metrics.rmse DESC"]
)

for run in runs:
    print(f"Run ID: {run.info.run_id}")
    print(f"RMSE: {run.data.metrics.get('rmse', 'N/A')}")
    print("---")

Run ID: 495e5822537b477f9abca08b0ebf5f93
RMSE: 4.62959559484091
---
Run ID: 5f22436317fb448a8ee98bb0f805b6a3
RMSE: 4.580404084480872
---


In [104]:
# Print the runs we found
print(f"Total runs found: {len(runs)}\n")

for run in runs:
    print(f"Run ID: {run.info.run_id}")
    print(f"Status: {run.info.status}")
    if run.data.metrics:
        print(f"Metrics: {run.data.metrics}")
    print("---")

Total runs found: 2

Run ID: 495e5822537b477f9abca08b0ebf5f93
Status: FINISHED
Metrics: {'rmse': 4.62959559484091}
---
Run ID: 5f22436317fb448a8ee98bb0f805b6a3
Status: FINISHED
Metrics: {'rmse': 4.580404084480872}
---


In [105]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 495e5822537b477f9abca08b0ebf5f93, rmse: 4.6296
run id: 5f22436317fb448a8ee98bb0f805b6a3, rmse: 4.5804


In [106]:
import mlflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [107]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.tpep_dropoff_datetime = pd.to_datetime(df.tpep_dropoff_datetime)
    df.tpep_pickup_datetime = pd.to_datetime(df.tpep_pickup_datetime)

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    df = df[((df.duration >= 1) & (df.duration <= 60))]
    categorical = ['PULocationID', 'DOLocationID']
    numerical = ['trip_distance']
    df[categorical] = df[categorical].astype(str)

    return df

In [108]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [109]:
run_id = "00ca1d080424417f92b0c45d36c2006a"
model_uri = f"runs:/{run_id}/model"

# First, let's check if this run exists and has a model
try:
    run = client.get_run(run_id)
    print(f"Run status: {run.info.status}")
    print(f"Artifacts: {run.data.tags}")
    # Try to get artifacts
    artifacts = client.list_artifacts(run_id)
    print(f"Logged artifacts: {[a.path for a in artifacts]}")
except Exception as e:
    print(f"Error fetching run: {e}")

# Now register if artifacts exist
mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

Run status: FINISHED
Artifacts: {'mlflow.user': 'PrashantKumar6', 'mlflow.source.name': 'trainamodel.ipynb', 'mlflow.source.type': 'NOTEBOOK', 'mlflow.runName': 'legendary-vole-126'}
Logged artifacts: ['feature_importance_weight.json', 'feature_importance_weight.png', 'preprocessor']


Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/04/20 14:09:27 WARNING mlflow.tracking._model_registry.fluent: Run with id 00ca1d080424417f92b0c45d36c2006a has no artifacts at artifact path 'model', registering model based on models:/m-4e37059b1fc1487081b29b1faa803d28 instead
Created version '9' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1776674367769, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1776674367769, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='00ca1d080424417f92b0c45d36c2006a', run_link=None, source='models:/m-4e37059b1fc1487081b29b1faa803d28', status='READY', status_message=None, tags={}, user_id=None, version=9, workspace='default'>

In [110]:
# Find a run with a logged model
from mlflow.entities import ViewType

runs_with_models = client.search_runs(
    experiment_ids=["1"], 
    run_view_type=ViewType.ACTIVE_ONLY, 
    max_results=10,
    order_by=["metrics.rmse DESC"]
)

print("Looking for runs with logged models...")
valid_run = None
for run in runs_with_models:
    artifacts = client.list_artifacts(run.info.run_id)
    artifact_paths = [a.path for a in artifacts]
    if 'model' in artifact_paths:
        valid_run = run
        print(f"✓ Found run with model: {run.info.run_id}")
        print(f"  RMSE: {run.data.metrics.get('rmse', 'N/A')}")
        print(f"  Artifacts: {artifact_paths}")
        break
    else:
        print(f"✗ Run {run.info.run_id} has artifacts: {artifact_paths}")

if valid_run:
    print(f"\n→ Use this run_id: {valid_run.info.run_id}")
else:
    print("\n⚠ No runs found with a logged model. Check trainamodel.ipynb execution.")

Looking for runs with logged models...
✗ Run 97d94d167c564dbbaf856c3efa96191c has artifacts: ['feature_importance_weight.json', 'feature_importance_weight.png', 'preprocessor']
✗ Run f9312a8bd78b4236906897adf5dcf637 has artifacts: ['feature_importance_weight.json', 'feature_importance_weight.png', 'preprocessor']
✗ Run 5d24bb34c7dd4d89ae58190970a7ad36 has artifacts: []
✗ Run 36304cd84c7f42c885cb127850e10570 has artifacts: []
✗ Run 10d24fc0262e4675815e00ce31425943 has artifacts: []
✗ Run 07697be3511641769a714775f85e8b4f has artifacts: ['feature_importance_weight.json', 'feature_importance_weight.png']
✗ Run e07287e8ff0649db9d48e42db95797e9 has artifacts: []
✗ Run 3ccdcecf5a574321a6001feb3bd28d99 has artifacts: []
✗ Run f0891755e91e4e8a89ac53a580fb6718 has artifacts: []
✗ Run 28e21a5467a14fc99fdbf2624000baec has artifacts: []

⚠ No runs found with a logged model. Check trainamodel.ipynb execution.


In [111]:
# Load yellow taxi 2023 data from local folder (OFFLINE)
print("Loading data from local folder...")

# Load training and validation data
df_train = read_dataframe('data/yellow_tripdata_2023-01.parquet')
df_val = read_dataframe('data/yellow_tripdata_2023-02.parquet')

print(f"✓ Training data: {df_train.shape[0]} rows")
print(f"✓ Validation data: {df_val.shape[0]} rows")
print(f"\nDataset info:")
print(f"  Train date range: {df_train['tpep_pickup_datetime'].min()} to {df_train['tpep_pickup_datetime'].max()}")
print(f"  Val date range: {df_val['tpep_pickup_datetime'].min()} to {df_val['tpep_pickup_datetime'].max()}")

Loading data from local folder...
✓ Training data: 3009173 rows
✓ Validation data: 2855951 rows

Dataset info:
  Train date range: 2022-10-25 00:42:10 to 2023-02-01 00:56:53
  Val date range: 2008-12-31 23:06:21 to 2023-03-07 13:01:28


In [112]:
def preprocess(df, dv):
    """Preprocess data for modeling"""
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    
    dicts = df[categorical + numerical].to_dict(orient='records')
    X = dv.transform(dicts)
    
    return X

In [113]:
# Setup DictVectorizer and fit on training data
from sklearn.feature_extraction import DictVectorizer

# Create and fit DictVectorizer on training data
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']

categorical = ['PU_DO']
numerical = ['trip_distance']

dv = DictVectorizer()
train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

print(f"Features created: {len(dv.feature_names_)}")
print(f"X_train shape: {X_train.shape}")

Features created: 21802
X_train shape: (3009173, 21802)


In [136]:
def test_model(name, stage, X_test, y_test):
    """Test a model from MLflow model registry"""
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    
    print(f"Model loaded: {name} ({stage})")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_test shape: {y_test.shape}")
    
    # Make predictions
    y_pred = model.predict(X_test)
    print(f"y_pred shape: {y_pred.shape}")
    
    # Handle different prediction formats
    if len(y_pred.shape) > 1 and y_pred.shape[1] == 1:
        y_pred = y_pred.ravel()
    
    # Ensure same length
    min_len = min(len(y_test), len(y_pred))
    y_test_aligned = y_test[:min_len]
    y_pred_aligned = y_pred[:min_len]
    
    # Calculate metrics
    rmse = root_mean_squared_error(y_test_aligned, y_pred_aligned)
    
    print(f"\nModel: {name}")
    print(f"Stage: {stage}")
    print(f"RMSE on test set: {rmse:.4f}")
    
    return rmse

In [131]:
# Check what stages the model has
model_name = "nyc-taxi-regressor"

try:
    latest_versions = client.get_latest_versions(name=model_name)
    print(f"Model: {model_name}")
    print("Available versions and stages:")
    for version in latest_versions:
        print(f"  Version {version.version}: Stage = {version.current_stage}")
    
    # If no Production version, transition the highest version to Production
    prod_versions = [v for v in latest_versions if v.current_stage == "Production"]
    
    if not prod_versions:
        print("\n⚠️ No Production version found!")
        # Find the best version to promote (prefer Staging, then None)
        staging_versions = [v for v in latest_versions if v.current_stage == "Staging"]
        if staging_versions:
            version_to_promote = max(staging_versions, key=lambda v: int(v.version))
        else:
            version_to_promote = max(latest_versions, key=lambda v: int(v.version))
        
        print(f"Transitioning version {version_to_promote.version} to Production...")
        client.transition_model_version_stage(
            name=model_name,
            version=version_to_promote.version,
            stage="Production",
            archive_existing_versions=False
        )
        print(f"✓ Version {version_to_promote.version} transitioned to Production")
    else:
        print(f"\n✓ Version {prod_versions[0].version} already in Production")
        
except Exception as e:
    print(f"Error: {e}")
    print("Model may not be registered. Check earlier cells.")

Model: nyc-taxi-regressor
Available versions and stages:
  Version 5: Stage = None
  Version 9: Stage = Staging

⚠️ No Production version found!
Transitioning version 9 to Production...
✓ Version 9 transitioned to Production


C:\Users\PrashantKumar6\AppData\Local\Temp\ipykernel_43808\762827768.py:5: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)
C:\Users\PrashantKumar6\AppData\Local\Temp\ipykernel_43808\762827768.py:24: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [114]:
model_name  = "nyc-taxi-regressor"

# First, check if model exists and what versions it has
try:
    latest_versions = client.get_latest_versions(name=model_name)
    print(f"Model '{model_name}' found with versions:")
    for version in latest_versions:
        print(f"  Version: {version.version}, Stage: {version.current_stage}, Run ID: {version.run_id}")
except Exception as e:
    print(f"Error fetching model: {e}")
    print("\nRegistered models:")
    models = client.search_registered_models()
    for model in models:
        print(f"  - {model.name}")

Model 'nyc-taxi-regressor' found with versions:
  Version: 9, Stage: None, Run ID: 00ca1d080424417f92b0c45d36c2006a
  Version: 8, Stage: Staging, Run ID: 00ca1d080424417f92b0c45d36c2006a


C:\Users\PrashantKumar6\AppData\Local\Temp\ipykernel_43808\2112055613.py:5: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [115]:
if latest_versions:
    # Transition only the first version to Staging
    version_num = latest_versions[0].version
    model_version = version_num
    new_stage = "Staging"
    try:
        client.transition_model_version_stage(
            name=model_name,
            version=model_version,
            stage="Staging",
            archive_existing_versions=False
        )
        print(f"✓ Transitioned version {version_num} to Staging")
    except Exception as e:
        print(f"Error transitioning stage: {e}")
else:
    print("⚠ No versions found for this model. Register the model first.")

✓ Transitioned version 9 to Staging


C:\Users\PrashantKumar6\AppData\Local\Temp\ipykernel_43808\3473236993.py:7: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [116]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"the model version {model_version} was transitioned to {new_stage}"
)

<ModelVersion: aliases=[], creation_timestamp=1776674367769, current_stage='Staging', deployment_job_state=None, description='the model version 9 was transitioned to Staging', last_updated_timestamp=1776674416168, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='00ca1d080424417f92b0c45d36c2006a', run_link=None, source='models:/m-4e37059b1fc1487081b29b1faa803d28', status='READY', status_message=None, tags={}, user_id=None, version=9, workspace='default'>

In [117]:
df = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')

In [118]:
model_name  = "nyc-taxi-regressor"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"Version: {version.version}, Stage: {version.current_stage}")

Version: 5, Stage: None
Version: 9, Stage: Staging


C:\Users\PrashantKumar6\AppData\Local\Temp\ipykernel_43808\3305129931.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [119]:
client.download_artifacts(run_id=run_id, path="preprocessor", dst_path=".")

'c:\\Users\\PrashantKumar6\\Documents\\vscode\\mlops-zoomcamp\\03-training\\experiment_tracking\\preprocessor'

In [120]:
import pickle

with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [121]:
X_test = preprocess(df_val, dv)

In [123]:
target = 'duration'
y_test = df[target].values

In [140]:
%time test_model(name=model_name, stage="Production", X_test=X_test, y_test=y_test)

Model loaded: nyc-taxi-regressor (Production)
X_test shape: (2855951, 13575)
y_test shape: (3009173,)
y_pred shape: (2855951,)

Model: nyc-taxi-regressor
Stage: Production
RMSE on test set: 17.9927
CPU times: total: 1min 33s
Wall time: 9.09 s


17.992658641192428

In [ ]:
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage="Production",
    archive_existing_versions=True)